# Preprocessing

In [7]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.data import save_features

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

train = pd.read_csv(RAW / "application_train.csv")
test = pd.read_csv(RAW / "application_test.csv")
y = train["TARGET"]
df = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True).copy()
df.shape

(356255, 121)

`365243` in `DAYS_*` = NA; `XNA`/`XAP` = NA (per competition host).

In [8]:
print(df["CODE_GENDER"].unique().tolist())
print(df.loc[df["DAYS_EMPLOYED"] > 0, "DAYS_EMPLOYED"].unique()[:5])
print((df["ORGANIZATION_TYPE"] == "XNA").sum())

['M', 'F', 'XNA']
[365243]
64648


In [9]:
df = df.assign(days_employed_anom=(df["DAYS_EMPLOYED"] == 365243).astype("int8"))
day_cols = [c for c in df.columns if c.startswith("DAYS_")]
df[day_cols] = df[day_cols].replace(365243, np.nan)
df["CODE_GENDER"] = df["CODE_GENDER"].replace("XNA", np.nan)
df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].replace("XNA", np.nan)
df["DAYS_LAST_PHONE_CHANGE"] = df["DAYS_LAST_PHONE_CHANGE"].replace(0, np.nan)

# Feature Engineering

## Iteration 1

Hand-crafted ratios (adapted from public Home Credit solutions + own).

In [10]:
non_child = df["CNT_FAM_MEMBERS"] - df["CNT_CHILDREN"]
df = df.assign(
    annuity_income_percentage=df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"],
    credit_to_income=df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"],
    credit_to_annuity=df["AMT_CREDIT"] / df["AMT_ANNUITY"],
    credit_to_goods=df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"],
    payment_rate=df["AMT_ANNUITY"] / df["AMT_CREDIT"],
    days_employed_percentage=df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"],
    car_to_birth_ratio=df["OWN_CAR_AGE"] / df["DAYS_BIRTH"],
    car_to_employ_ratio=df["OWN_CAR_AGE"] / df["DAYS_EMPLOYED"],
    phone_to_birth_ratio=df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"],
    phone_to_employ_ratio=df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_EMPLOYED"],
    income_per_child=df["AMT_INCOME_TOTAL"] / (1 + df["CNT_CHILDREN"]),
    income_per_person=df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"],
    children_ratio=df["CNT_CHILDREN"] / df["CNT_FAM_MEMBERS"],
    cnt_non_child=non_child,
    child_to_non_child_ratio=df["CNT_CHILDREN"] / non_child,
    income_per_non_child=df["AMT_INCOME_TOTAL"] / non_child,
    credit_per_person=df["AMT_CREDIT"] / df["CNT_FAM_MEMBERS"],
    credit_per_child=df["AMT_CREDIT"] / (1 + df["CNT_CHILDREN"]),
    credit_per_non_child=df["AMT_CREDIT"] / non_child,
    long_employment=(df["DAYS_EMPLOYED"] < -2000).astype("int8"),
    retirement_age=(df["DAYS_BIRTH"] < -14000).astype("int8"),
).replace([np.inf, -np.inf], np.nan)
df.shape

(356255, 143)

## Iteration 4

In [11]:
df = df.assign(
    disposable_income=df["AMT_INCOME_TOTAL"] - df["AMT_ANNUITY"],
    disposable_income_ratio=(df["AMT_INCOME_TOTAL"] - df["AMT_ANNUITY"]) / df["AMT_INCOME_TOTAL"],
    disposable_per_person=(df["AMT_INCOME_TOTAL"] - df["AMT_ANNUITY"]) / df["CNT_FAM_MEMBERS"],
    annuity_per_person=df["AMT_ANNUITY"] / df["CNT_FAM_MEMBERS"],
    annuity_to_goods=df["AMT_ANNUITY"] / df["AMT_GOODS_PRICE"],
    goods_to_income=df["AMT_GOODS_PRICE"] / df["AMT_INCOME_TOTAL"],
    credit_markup=df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"],
    credit_markup_ratio=(df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]) / df["AMT_GOODS_PRICE"],
    income_to_age=df["AMT_INCOME_TOTAL"] / -df["DAYS_BIRTH"],
    income_to_employ=df["AMT_INCOME_TOTAL"] / -df["DAYS_EMPLOYED"],
).replace([np.inf, -np.inf], np.nan)
df.shape

(356255, 153)

# Save

In [12]:
ids = df["SK_ID_CURR"]
feat = pd.get_dummies(df.drop(columns=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]), dummy_na=True, dtype="int8").astype("float32")
out = pd.concat([ids, feat], axis=1)
save_features(out, "application")
out.shape

(356255, 288)